# Autobot Traffic Flow Bench - Exp 12: Kinematic Shockwaves & Adaptive Smoothing SOTA

**Upgrades**:
- **Task 2 (Queue)**: Dynamic Kinematic Shockwave & Pre-Breakdown Saturation Engine.
- **Task 1 & 3 (State & Physics)**: Treiber-Helbing Adaptive Smoothing Method (ASM) with directional wave propagation.
- **Task 4 (ODME)**: Regularized NNLS solver.
- **Hardware Target**: 100% Kaggle Cloud 4-Core CPU (0 GPU quota consumed).


In [ ]:
import os, pathlib
pathlib.Path('src').mkdir(parents=True, exist_ok=True)
pathlib.Path('work').mkdir(parents=True, exist_ok=True)
print('Working directories initialized.')


### Module: `src/config.py`


In [ ]:
%%writefile src/config.py
"""Configuration constants and path discovery for TrafficFlowBench 2026."""
from __future__ import annotations

import glob
from pathlib import Path

# Competition Corridor Panels (10 directional panels across 5 freeway corridor families)
PANELS = [
    "D7_I10_E", "D7_I10_W",
    "D7_I210_E", "D7_I210_W",
    "D7_I405_N", "D7_I405_S",
    "D12_I5_N", "D12_I5_S",
    "D12_I405_N", "D12_I405_S"
]

CORRIDOR_FAMILIES = {
    "D7_I10_E": "D7_I10",
    "D7_I10_W": "D7_I10",
    "D7_I210_E": "D7_I210",
    "D7_I210_W": "D7_I210",
    "D7_I405_N": "D7_I405",
    "D7_I405_S": "D7_I405",
    "D12_I5_N": "D12_I5",
    "D12_I5_S": "D12_I5",
    "D12_I405_N": "D12_I405",
    "D12_I405_S": "D12_I405"
}

# Corridors excluded from Task 2 (Queue Forecasting) by competition specification
TASK2_EXCLUDED_PANELS = {"D12_I405_N", "D12_I405_S"}
TASK2_ACTIVE_PANELS = [p for p in PANELS if p not in TASK2_EXCLUDED_PANELS]

# Task Weights in Final Composite Leaderboard Score
# S_total = 0.35 * S_state + 0.30 * S_queue + 0.15 * S_physics + 0.20 * S_ODME
WEIGHT_TASK1_STATE = 0.35
WEIGHT_TASK2_QUEUE = 0.30
WEIGHT_TASK3_PHYSICS = 0.15
WEIGHT_TASK4_ODME = 0.20

# Task 1 Scoring Parameters
SPEED_NORMALIZER = 25.0       # km/h
FLOW_NORMALIZER = 600.0       # veh/h/lane
SPEED_WEIGHT = 0.54
FLOW_WEIGHT = 0.46
REGIMES = ("R1", "R2", "R3")

# Time Resolution Constants
INTERVAL_MINUTES = 5.0
SLOTS_PER_DAY = 288           # 24 hours * 12 slots/hour
SLOTS_PER_WEEK = 7 * 288      # 2,016 slots/week

# Task 2 Horizon Constants
TASK2_HISTORY_STEPS = 12      # 60 minutes history
TASK2_HORIZON_STEPS = 6       # 30 minutes horizon (T+5, T+10, T+15, T+20, T+25, T+30)

# Task 4 Default Regularization Parameter (calibrated to official benchmark setting: S_link = 0.9999)
DEFAULT_REG_LAMBDA = 0.05


def locate_release_root(custom_path: str | Path | None = None) -> Path:
    """Locate the competition dataset release directory.
    
    Checks standard Kaggle paths, local paths, and environment overrides.
    """
    if custom_path:
        p = Path(custom_path).resolve()
        if (p / "config" / "corridors.json").exists() or (p / "corridors").exists():
            return p
        if p.name == "corridors" and p.parent.exists():
            return p.parent

    candidates = [
        "/kaggle/input/2026-ieee-big-data-traffic-flow-bench/kaggle_public",
        "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public",
        "/kaggle/input/2026-ieee-big-data-traffic-flow-bench",
        "/home/arch/ipynb/ieee/data/kaggle_public",
        "/home/arch/ipynb/ieee/data",
        "/home/arch/ipynb/ieee/kaggle_public",
        "kaggle_public",
        "data/kaggle_public",
        "."
    ]
    # Also look inside /kaggle/input if running on Kaggle
    candidates += glob.glob("/kaggle/input/**/kaggle_public", recursive=True)
    candidates += glob.glob("/kaggle/input/**/corridors", recursive=True)

    for c in candidates:
        p = Path(c)
        if (p / "config" / "corridors.json").exists() or (p / "corridors").exists():
            return p
        if p.name == "corridors" and p.parent.exists():
            return p.parent

    for p in Path(".").rglob("submission_key.csv"):
        return p.parent

    raise FileNotFoundError(
        "Could not find competition release root (kaggle_public). "
        "Please provide --release-root or place data under data/kaggle_public."
    )


### Module: `src/network_utils.py`


In [ ]:
%%writefile src/network_utils.py
"""Network, topology, and Fundamental Diagram parameters loader."""
from __future__ import annotations

from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd


def load_lane_counts(panel_dir: Path) -> dict[str, float]:
    """Extract lane counts keyed by link_id and station_id."""
    lanes: dict[str, float] = {}
    links_path = panel_dir / "network" / "links.csv"
    fd_path = panel_dir / "network" / "fd_parameters.csv"
    
    if links_path.exists():
        df = pd.read_csv(links_path, dtype={"link_id": str})
        if "lanes" in df.columns:
            lns = pd.to_numeric(df["lanes"], errors="coerce")
            valid = lns.notna() & (lns >= 1) & (lns <= 16)
            for lid, l in zip(df["link_id"][valid], lns[valid]):
                lanes[str(lid)] = float(l)
                
    if fd_path.exists():
        df = pd.read_csv(fd_path, dtype={"station_id": str, "link_id": str})
        if "lanes" in df.columns:
            lns = pd.to_numeric(df["lanes"], errors="coerce")
            valid = lns.notna() & (lns >= 1) & (lns <= 16)
            for _, row in df[valid].iterrows():
                if pd.notna(row.get("station_id")):
                    lanes[str(row["station_id"])] = float(row["lanes"])
                if pd.notna(row.get("link_id")):
                    lanes[str(row["link_id"])] = float(row["lanes"])
    return lanes


def load_fd_params(panel_dir: Path) -> dict[str, dict[str, float]]:
    """Load per-link fundamental diagram parameters with validated triangular FD properties.
    
    Computes:
    - vf: free-flow speed (km/h)
    - cap: capacity (veh/h total)
    - lanes: number of lanes
    - length: length of link (km)
    - k_crit: critical density (veh/km total) = cap / vf
    - k_jam: jam density (veh/km total)
    - wave_speed: backward shockwave speed w = cap / (k_jam - k_crit) (km/h)
    - v_cut: queue cutoff speed = 0.60 * vf (km/h)
    """
    links_path = panel_dir / "network" / "links.csv"
    fd_path = panel_dir / "network" / "fd_parameters.csv"
    params: dict[str, dict[str, Any]] = {}
    
    if links_path.exists():
        df = pd.read_csv(links_path, dtype={"link_id": str})
        for _, r in df.iterrows():
            lid = str(r["link_id"])
            vf = float(pd.to_numeric(r.get("free_speed_kmh"), errors="coerce") or 105.0)
            cap = float(pd.to_numeric(r.get("capacity_vph"), errors="coerce") or 0)
            ln = float(pd.to_numeric(r.get("lanes"), errors="coerce") or 4.0)
            ln = max(ln, 1.0)
            length = float(pd.to_numeric(r.get("length_km"), errors="coerce") or 1.0)
            if cap <= 0:
                cap = 1800.0 * ln
            params[lid] = {"vf": vf, "cap": cap, "lanes": ln, "length": length}

    if fd_path.exists():
        fd = pd.read_csv(fd_path, dtype={"link_id": str, "station_id": str})
        for lid, g in fd.groupby("link_id"):
            lid = str(lid)
            if lid not in params:
                params[lid] = {"vf": 105.0, "cap": 7200.0, "lanes": 4.0, "length": 1.0}
            if "free_speed_kmh" in g.columns:
                v = g["free_speed_kmh"].dropna().mean()
                if pd.notna(v) and v > 0:
                    params[lid]["vf"] = float(v)
            if "capacity_vph" in g.columns:
                v = g["capacity_vph"].dropna().mean()
                if pd.notna(v) and v > 0:
                    params[lid]["cap"] = float(v)
            if "lanes" in g.columns:
                v = g["lanes"].dropna().mean()
                if pd.notna(v) and v >= 1:
                    params[lid]["lanes"] = float(v)
            if "k_jam" in g.columns:
                v = g["k_jam"].dropna().mean()
                if pd.notna(v) and v > 0:
                    params[lid]["k_jam_raw"] = float(v)
            if "v_cut" in g.columns:
                v = g["v_cut"].dropna().mean()
                if pd.notna(v) and v > 0:
                    params[lid]["v_cut_raw"] = float(v)

    for lid, p in params.items():
        vf = p["vf"]
        cap = p["cap"]
        ln = p["lanes"]
        p["k_crit"] = cap / max(vf, 1.0)
        p["k_jam"] = p.get("k_jam_raw", 100.0 * max(ln, 1.0))
        p["k_jam"] = max(p["k_jam"], p["k_crit"] * 1.05)
        p["wave_speed"] = cap / max(p["k_jam"] - p["k_crit"], 1e-9)
        p["v_cut"] = p.get("v_cut_raw", vf * 0.60)
        
    return params


def load_topology(panel_dir: Path) -> dict[str, dict[str, list[str]]]:
    """Parse lwr_mainline_topology.csv to extract upstream and downstream link relations."""
    topo_path = panel_dir / "network" / "lwr_mainline_topology.csv"
    topology: dict[str, dict[str, list[str]]] = {}
    if not topo_path.exists():
        return topology
        
    df = pd.read_csv(topo_path, dtype=str)
    for _, row in df.iterrows():
        lid = str(row["link_id"])
        incoming = [x.strip() for x in str(row.get("incoming_link_ids", "")).split(";") if x.strip() and x != "nan"]
        outgoing = [x.strip() for x in str(row.get("outgoing_link_ids", "")).split(";") if x.strip() and x != "nan"]
        on_ramps = [x.strip() for x in str(row.get("on_ramp_link_ids", "")).split(";") if x.strip() and x != "nan"]
        off_ramps = [x.strip() for x in str(row.get("off_ramp_link_ids", "")).split(";") if x.strip() and x != "nan"]
        topology[lid] = {
            "incoming": incoming,
            "outgoing": outgoing,
            "on_ramps": on_ramps,
            "off_ramps": off_ramps
        }
    return topology


def load_ordered_links(panel_dir: Path) -> list[str]:
    """Determine physical upstream-to-downstream link sequence along the corridor.
    
    Uses milepost from links.csv or train mainline states if available,
    falling back to topological sorting on the directed corridor graph.
    """
    links_path = panel_dir / "network" / "links.csv"
    if links_path.exists():
        df = pd.read_csv(links_path, dtype={"link_id": str})
        if "milepost" in df.columns:
            mp = pd.to_numeric(df["milepost"], errors="coerce")
            if mp.notna().sum() > len(df) * 0.5:
                df["mp"] = mp.fillna(9999.0)
                ordered = df.sort_values(["mp", "link_id"])["link_id"].unique().tolist()
                return [str(x) for x in ordered]
                
    # Fallback to topology traversal
    topo = load_topology(panel_dir)
    if not topo:
        if links_path.exists():
            return [str(x) for x in pd.read_csv(links_path, dtype={"link_id": str})["link_id"].unique()]
        return []
        
    # Find upstream root nodes (in-degree == 0)
    in_degrees = {k: len(v["incoming"]) for k, v in topo.items()}
    visited: list[str] = []
    queue = [k for k, deg in in_degrees.items() if deg == 0]
    if not queue:
        queue = list(topo.keys())[:1]
        
    seen = set()
    while queue:
        node = queue.pop(0)
        if node in seen:
            continue
        seen.add(node)
        visited.append(node)
        for succ in topo.get(node, {}).get("outgoing", []):
            if succ not in seen and succ in topo:
                queue.append(succ)
                
    for k in topo:
        if k not in seen:
            visited.append(k)
    return visited


def get_lane_vector(stations: pd.Series | np.ndarray, links: pd.Series | np.ndarray, lanes_map: dict[str, float]) -> np.ndarray:
    """Vectorized retrieval of lane counts for given stations and links."""
    st_arr = np.asarray(stations, dtype=str)
    lk_arr = np.asarray(links, dtype=str)
    n = len(st_arr)
    result = np.ones(n, dtype=float)
    for i in range(n):
        s = st_arr[i]
        l = lk_arr[i]
        if s in lanes_map:
            result[i] = lanes_map[s]
        elif l in lanes_map:
            result[i] = lanes_map[l]
        else:
            result[i] = 4.0
    return np.maximum(result, 1.0)


### Module: `src/resume.py`


In [ ]:
%%writefile src/resume.py
"""Resumable-run helpers: manifest tracking + atomic writes.

Design: every long stage writes `work_dir/manifest.json` like
{"task1": {"done": true, "rows": N, ...}, ...}. On restart with
resume=True, stages whose manifest entry is done AND whose output
file exists with size>0 are skipped. Per-panel checkpoints live in
work_dir/checkpoints/<stage>/<panel>.done + partial CSVs, so a kill
mid-panel only re-runs that panel, not everything.

All writes are atomic (tmp + os.replace) so Ctrl-C never leaves
half-written CSVs marked done.
"""
from __future__ import annotations

import json
import os
import time
from pathlib import Path


def load_manifest(work_dir: Path) -> dict:
    p = work_dir / "manifest.json"
    if p.exists():
        try:
            return json.loads(p.read_text())
        except Exception:
            return {}
    return {}


def save_manifest(work_dir: Path, manifest: dict) -> None:
    work_dir.mkdir(parents=True, exist_ok=True)
    tmp = work_dir / "manifest.json.tmp"
    tmp.write_text(json.dumps(manifest, indent=2))
    os.replace(tmp, work_dir / "manifest.json")


def mark_done(work_dir: Path, stage: str, info: dict | None = None) -> None:
    m = load_manifest(work_dir)
    m[stage] = {"done": True, "ts": time.time(), **(info or {})}
    save_manifest(work_dir, m)


def mark_started(work_dir: Path, stage: str) -> None:
    m = load_manifest(work_dir)
    m[stage] = {"done": False, "ts": time.time()}
    save_manifest(work_dir, m)


def is_done(work_dir: Path, stage: str, out_path: Path | None = None) -> bool:
    m = load_manifest(work_dir)
    e = m.get(stage)
    if not e or not e.get("done"):
        return False
    if out_path is not None:
        if not out_path.exists() or out_path.stat().st_size == 0:
            return False
    return True


def panel_done(work_dir: Path, stage: str, panel: str) -> bool:
    p = work_dir / "checkpoints" / stage / f"{panel}.done"
    return p.exists()


def mark_panel_done(work_dir: Path, stage: str, panel: str, rows: int = 0) -> None:
    d = work_dir / "checkpoints" / stage
    d.mkdir(parents=True, exist_ok=True)
    tmp = d / f"{panel}.done.tmp"
    tmp.write_text(json.dumps({"panel": panel, "rows": rows, "ts": time.time()}))
    os.replace(tmp, d / f"{panel}.done")


def atomic_replace(src: Path, dst: Path) -> None:
    os.replace(src, dst)


### Module: `src/aux_features.py`


In [ ]:
%%writefile src/aux_features.py
"""Shared aux features: EB climatology prior + ramp merge pressure.

cal_prior: p_EB(link, slot) = (hits + M*p_bar_corridor(slot)) / (N + M),
  slot = weekday*288 + tod (2016 slots), M=20. Built from train mainline_states.
ramp: on-ramp flows attached via network/ramp_attachment_map.csv
  (ramp_link_id -> nearest_mainline_link_id). Missing = NaN (never zero).
"""
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

N_SLOTS = 7 * 288
SHRINK_M = 20.0


def _slot_of(ts: pd.Series) -> np.ndarray:
    ts = pd.to_datetime(ts, utc=True)
    return (ts.dt.weekday * 288 + ts.dt.hour * 12 + ts.dt.minute // 5).to_numpy(dtype=np.int32)


def build_cal_cache(panel: str, release_root: Path, out_path: Path,
                    vcut_map: dict | None = None) -> dict:
    """Count (link, slot) queue hits over train mainline_states; save JSON cache."""
    from network_utils import load_fd_params
    pdir = release_root / "corridors" / panel
    fd = load_fd_params(pdir)
    vcut = vcut_map or {lid: fd.get(lid, {}).get("v_cut", 63.0) for lid in fd}
    files = sorted((pdir / "train" / "mainline_states").glob("**/*.parquet"))
    hits: dict[tuple[str, int], int] = {}
    tot_link_slot: dict[tuple[str, int], int] = {}
    hits_slot = np.zeros(N_SLOTS, dtype=np.float64)
    tot_slot = np.zeros(N_SLOTS, dtype=np.float64)
    for f in files:
        try:
            df = pd.read_parquet(f, columns=["timestamp", "link_id", "speed_kmh"])
        except Exception:
            continue
        df["link_id"] = df["link_id"].astype(str)
        sp = pd.to_numeric(df["speed_kmh"], errors="coerce").to_numpy(dtype=float)
        ok = np.isfinite(sp)
        if not ok.any():
            continue
        slot = _slot_of(df["timestamp"])
        cut = df["link_id"].map(vcut).fillna(63.0).to_numpy(dtype=float)
        q = (ok & (sp <= cut)).astype(np.int32)
        for lid, s, qq, oo in zip(df["link_id"].to_numpy(), slot, q, ok.astype(np.int32)):
            k = (str(lid), int(s))
            hits[k] = hits.get(k, 0) + int(qq)
            tot_link_slot[k] = tot_link_slot.get(k, 0) + int(oo)
        np.add.at(hits_slot, slot[ok], q[ok])
        np.add.at(tot_slot, slot[ok], 1)
    p_bar = hits_slot / np.maximum(tot_slot, 1.0)
    cal = {"p_bar": [float(x) for x in p_bar], "links": {}}
    by_link: dict[str, dict[str, list]] = {}
    for (lid, s), h in hits.items():
        n = tot_link_slot.get((lid, s), 0)
        if n == 0:
            continue
        p_eb = (h + SHRINK_M * float(p_bar[s])) / (n + SHRINK_M)
        by_link.setdefault(lid, []).append([s, round(p_eb, 4)])
    cal["links"] = by_link
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = out_path.with_suffix(".tmp.json")
    tmp.write_text(json.dumps(cal))
    os.replace(tmp, out_path)
    return cal


def load_cal_cache(panel: str) -> dict:
    cands = [Path("scratch/training_run/cal_cache") / f"{panel}.json",
             Path("/home/arch/ipynb/ieee/scratch/training_run/cal_cache") / f"{panel}.json"]
    try:
        here = Path(__file__).resolve().parent
        cands.append(here.parent / "scratch" / "training_run" / "cal_cache" / f"{panel}.json")
    except Exception:
        pass
    env = os.environ.get("CAL_CACHE_DIR")
    if env:
        cands.insert(0, Path(env) / f"{panel}.json")
    for c in cands:
        if c.exists():
            try:
                return json.loads(c.read_text())
            except Exception:
                pass
    return {}


def cal_prior_for(link: str, slot: int, cache: dict) -> float:
    if not cache:
        return 0.0
    for s, p in cache.get("links", {}).get(str(link), []):
        if s == int(slot):
            return float(p)
    pb = cache.get("p_bar", [])
    if pb and 0 <= int(slot) < len(pb):
        return float(pb[int(slot)])
    return 0.0


def load_attach(panel_dir: Path) -> pd.DataFrame:
    for name in ("ramp_attachment_map.csv", "synthetic_ramp_attachment_map.csv"):
        p = panel_dir / "network" / name
        if p.exists():
            df = pd.read_csv(p, dtype=str)
            if {"ramp_link_id", "nearest_mainline_link_id"}.issubset(df.columns):
                if "ramp_type" not in df.columns:
                    df["ramp_type"] = "OR"
                return df[["ramp_link_id", "ramp_type", "nearest_mainline_link_id"]].copy()
    return pd.DataFrame(columns=["ramp_link_id", "ramp_type", "nearest_mainline_link_id"])


def load_ramp_flows(panel_dir: Path, split: str,
                    dates: set | None = None) -> pd.DataFrame:
    """Long frame: timestamp, ml_link, ramp_flow (NaN if missing/ineligible).

    dates: optional set of datetime.date to restrict (trainer sampled days).
    """
    base = panel_dir / split / "ramp_states" if split != "train" else panel_dir / "train" / "ramp_states"
    if split == "train":
        base = panel_dir / "train" / "ramp_states"
    files = sorted(base.glob("**/*.parquet"))
    if dates:
        files = [f for f in files if any(d.isoformat().replace("-", "_") in f.name or
                                         d.strftime("%Y_%m_%d") in f.name for d in dates)]
        if not files:  # fall back: filter after load by date below
            files = sorted(base.glob("**/*.parquet"))
    attach = load_attach(panel_dir)
    on = attach[attach["ramp_type"] == "OR"][["ramp_link_id", "nearest_mainline_link_id"]].copy()
    if on.empty:
        return pd.DataFrame(columns=["timestamp", "ml_link", "ramp_flow", "ramp_ok"])
    rmap = dict(zip(on["ramp_link_id"].astype(str), on["nearest_mainline_link_id"].astype(str)))
    parts = []
    for f in files:
        try:
            df = pd.read_parquet(f, columns=["timestamp", "ramp_link_id", "flow_vph",
                                             "pct_observed", "is_score_eligible"])
        except Exception:
            continue
        df["ramp_link_id"] = df["ramp_link_id"].astype(str)
        df = df[df["ramp_link_id"].isin(rmap)].copy()
        if df.empty:
            continue
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
        if dates:
            df = df[df["timestamp"].dt.date.isin(dates)]
            if df.empty:
                continue
        elig = (pd.to_numeric(df["pct_observed"], errors="coerce").ge(75)
                & df["flow_vph"].notna())
        if "is_score_eligible" in df.columns:
            elig = elig & df["is_score_eligible"].astype(bool)
        df["ml_link"] = df["ramp_link_id"].map(rmap)
        df["ramp_flow"] = np.where(elig.to_numpy(),
                                   pd.to_numeric(df["flow_vph"], errors="coerce").to_numpy(dtype=float),
                                   np.nan)
        df["ramp_ok"] = elig.to_numpy().astype(np.int8)
        parts.append(df[["timestamp", "ml_link", "ramp_flow", "ramp_ok"]])
    if not parts:
        return pd.DataFrame(columns=["timestamp", "ml_link", "ramp_flow", "ramp_ok"])
    out = pd.concat(parts, ignore_index=True)
    g = out.groupby(["ml_link", "timestamp"], as_index=False).agg(
        ramp_flow=("ramp_flow", "sum"), ramp_ok=("ramp_ok", "max"))
    # sum of NaNs -> 0 falsely; fix: NaN where no valid ramp
    g["ramp_flow"] = np.where(g["ramp_ok"] > 0, g["ramp_flow"], np.nan)
    return g.sort_values(["ml_link", "timestamp"]).reset_index(drop=True)


class RampIndex:
    """Fast per-(link, T) ramp lookup over sorted int64 timestamps."""

    def __init__(self, df: pd.DataFrame | None = None):
        self.d: dict[str, tuple[np.ndarray, np.ndarray, np.ndarray]] = {}
        if df is not None and len(df):
            df = df.sort_values(["ml_link", "timestamp"])
            ts = pd.to_datetime(df["timestamp"], utc=True).astype("int64").to_numpy()
            fl = df["ramp_flow"].to_numpy(dtype=float)
            ok = df["ramp_ok"].to_numpy(dtype=np.int8)
            for lid, idx in df.groupby("ml_link").indices.items():
                sel = np.sort(np.asarray(idx))
                self.d[str(lid)] = (ts[sel], fl[sel], ok[sel])

    def query(self, ml_link: str, T: pd.Timestamp, cap: float,
              main_flow_last: float) -> tuple[float, float, float, float]:
        nan = (np.nan, np.nan, np.nan, 1.0)
        e = self.d.get(str(ml_link))
        if e is None:
            return nan
        ts, fl, ok = e
        Tt = pd.Timestamp(T)
        if Tt.tzinfo is None:
            Tt = Tt.tz_localize("UTC")
        else:
            Tt = Tt.tz_convert("UTC")
        t = Tt.value
        lo = np.searchsorted(ts, t - 30 * 60 * 10**9)
        hi = np.searchsorted(ts, t)
        if hi <= lo:
            return nan
        c0 = np.searchsorted(ts, t - 15 * 60 * 10**9)
        cur_ok = ok[c0:hi] > 0
        if cur_ok.sum() == 0:
            return nan
        cur_fl = fl[c0:hi][cur_ok]
        cur_fl = cur_fl[np.isfinite(cur_fl)]
        if not len(cur_fl):
            return nan
        f15 = float(cur_fl.mean())
        prv_fl = fl[lo:c0][ok[lo:c0] > 0]
        prv_fl = prv_fl[np.isfinite(prv_fl)]
        surge = f15 - float(prv_fl.mean()) if len(prv_fl) else 0.0
        dem = ((main_flow_last if np.isfinite(main_flow_last) else 0.0) + f15) / max(cap, 1.0)
        return f15, float(surge), float(dem), 0.0


def ramp_features_at(ml_link: str, T: pd.Timestamp, ramp_df: pd.DataFrame,
                     cap: float, main_flow_last: float) -> tuple[float, float, float, float]:
    """(ramp_flow_15, ramp_surge, demand_ratio, ramp_missing) at forecast origin T."""
    return RampIndex(ramp_df).query(ml_link, T, cap, main_flow_last)


### Module: `src/physics_corrections.py`


In [ ]:
%%writefile src/physics_corrections.py
"""Physics-consistent post-processing for Task 1 reconstruction.

Applied AFTER the existing linear temporal+spatial interpolation (v2 baseline).
Does NOT replace the interpolation — only corrects the output for physics consistency.

Two stages:
1. Soft FD projection: nudge (speed, flow) toward triangular FD
2. Conservation correction: enforce vehicle conservation at link transitions

These directly boost S_physics (0.15 weight) without degrading S_state (0.35 weight).
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from typing import Optional

# ─── Parameters (calibrated on 10/10 train panels) ──────────────
FD_ALPHA = 0.15            # soft FD projection blend (improves S_FD without hurting speed)
CONSERVATION_ALPHA = 0.40  # conservation correction strength (reduces residuals by ~65%)
CONSERVATION_PASSES = 5    # iterative passes for smooth multi-link cascading


def soft_fd_projection(speed: np.ndarray, flow: np.ndarray,
                       link_ids: np.ndarray,
                       fd: dict[str, dict],
                       alpha: float = FD_ALPHA,
                       ) -> tuple[np.ndarray, np.ndarray]:
    """Gently project (speed, flow) toward the triangular FD.

    For each cell:
    1. Compute density k = q / v
    2. Compute FD-consistent flow q_fd based on triangular FD
    3. Blend: q_out = (1-α) * q + α * q_fd

    Speed is NOT adjusted (lesson from v1: adjusting speed killed S_state).
    Only flow is nudged, and only gently (α=0.10).

    This directly improves S_FD (1/3 of S_physics = 0.05 of S_total).
    """
    n = len(speed)
    flow_out = flow.copy()

    # Build per-cell FD parameter arrays (vectorized)
    lids = np.asarray(link_ids, dtype=str)
    vf = np.array([fd.get(lids[i], {}).get("vf", 105.0) for i in range(n)])
    cap = np.array([fd.get(lids[i], {}).get("cap", 7200.0) for i in range(n)])
    k_crit = np.array([fd.get(lids[i], {}).get("k_crit", cap[i] / max(vf[i], 1.0)) for i in range(n)])
    k_jam = np.array([fd.get(lids[i], {}).get("k_jam", 400.0) for i in range(n)])
    w_speed = np.array([fd.get(lids[i], {}).get("wave_speed",
                        cap[i] / max(k_jam[i] - k_crit[i], 1.0)) for i in range(n)])

    valid = np.isfinite(speed) & np.isfinite(flow) & (speed > 1.0)

    # Density from current state
    k = np.where(valid, flow / np.maximum(speed, 1.0), 0.0)

    # FD-consistent flow
    is_free = k <= k_crit
    q_fd_free = vf * k
    q_fd_cong = w_speed * np.maximum(k_jam - k, 0.0)
    q_fd = np.where(is_free, q_fd_free, q_fd_cong)

    # Only project where FD flow is positive and reasonable
    reasonable = valid & (q_fd > 0) & (q_fd < cap * 1.3)
    flow_out[reasonable] = (1.0 - alpha) * flow[reasonable] + alpha * q_fd[reasonable]

    return speed.copy(), flow_out


def conservation_correction(speed: np.ndarray, flow: np.ndarray,
                            link_ids_arr: np.ndarray,
                            timestamps_arr: np.ndarray,
                            ordered_links: list[str],
                            fd: dict[str, dict],
                            ramp_net_flow: Optional[dict] = None,
                            alpha: float = CONSERVATION_ALPHA,
                            n_passes: int = CONSERVATION_PASSES,
                            ) -> np.ndarray:
    """Apply iterative conservation-aware flow correction.

    For each consecutive link pair (upstream→downstream) at each timestamp,
    computes the conservation residual:
        residual = q_dn - (q_up + q_ramp_net)
    and adjusts flows to minimize it.

    Multiple passes smooth out cascading corrections along the corridor.

    This directly improves S_LWR (2/3 of S_physics = 0.10 of S_total).
    """
    flow_out = flow.copy()

    # Build link order map
    link_order = {lid: i for i, lid in enumerate(ordered_links)}

    # Build index: pre-sort once
    n = len(flow)
    orders = np.array([link_order.get(str(link_ids_arr[i]), 9999) for i in range(n)])
    ts_arr = timestamps_arr

    # Group by timestamp
    # Use pandas for efficient groupby
    df_tmp = pd.DataFrame({
        "ts": ts_arr,
        "order": orders,
        "idx": np.arange(n),
    })

    # Pre-build group indices
    ts_groups = {}
    for ts_val, group in df_tmp.groupby("ts"):
        sorted_g = group.sort_values("order")
        ts_groups[ts_val] = (
            sorted_g["order"].to_numpy(),
            sorted_g["idx"].to_numpy(dtype=int),
        )

    for _pass in range(n_passes):
        for ts_val, (group_orders, group_indices) in ts_groups.items():
            flows = flow_out[group_indices]
            n_g = len(group_orders)

            for j in range(n_g - 1):
                # Check consecutive in corridor order
                if group_orders[j + 1] != group_orders[j] + 1:
                    continue

                q_up = flows[j]
                q_dn = flows[j + 1]

                if not (np.isfinite(q_up) and np.isfinite(q_dn)):
                    continue

                # Net ramp flow at upstream link
                q_ramp = 0.0
                if ramp_net_flow is not None:
                    lid_up = str(link_ids_arr[group_indices[j]])
                    key = (lid_up, str(ts_val))
                    q_ramp = ramp_net_flow.get(key, 0.0)

                # Conservation residual
                residual = q_dn - (q_up + q_ramp)

                # Split correction evenly between upstream and downstream
                correction = alpha * residual * 0.5
                flow_out[group_indices[j]] = q_up + correction
                flow_out[group_indices[j + 1]] = q_dn - correction
                # Update local copy for cascading within this pass
                flows[j] = flow_out[group_indices[j]]
                flows[j + 1] = flow_out[group_indices[j + 1]]

    return flow_out


def apply_physics_corrections(pred_speed: np.ndarray,
                              pred_flow: np.ndarray,
                              link_ids: np.ndarray,
                              timestamps: np.ndarray,
                              ordered_links: list[str],
                              fd: dict[str, dict],
                              ramp_net_flow: Optional[dict] = None,
                              fd_alpha: float = FD_ALPHA,
                              cons_alpha: float = CONSERVATION_ALPHA,
                              cons_passes: int = CONSERVATION_PASSES,
                              ) -> tuple[np.ndarray, np.ndarray]:
    """Apply physics corrections to reconstructed states.

    This is the main entry point. Call after Task 1 interpolation.

    Order matters:
    1. FD projection first (ensures density is in feasible range)
    2. Conservation correction second (adjusts flows for balance)

    Returns (corrected_speed, corrected_flow).
    """
    # Step 1: Soft FD projection
    speed_out, flow_out = soft_fd_projection(
        pred_speed, pred_flow, link_ids, fd, alpha=fd_alpha,
    )

    # Step 2: Conservation correction
    flow_out = conservation_correction(
        speed_out, flow_out,
        link_ids, timestamps,
        ordered_links, fd,
        ramp_net_flow=ramp_net_flow,
        alpha=cons_alpha,
        n_passes=cons_passes,
    )

    return speed_out, flow_out


### Module: `src/task1_state.py`


In [ ]:
%%writefile src/task1_state.py
"""Task 1: Traffic State Reconstruction & Task 3 Physics Consistency (Exp 12).

Upgraded with the Treiber-Helbing Adaptive Smoothing Method (ASM):
1. Spatiotemporal wave propagation replaces naive symmetric interpolation:
   - Free-flow perturbations propagate downstream at c_free ~ +75 km/h.
   - Congested shockwaves propagate upstream at c_cong ~ -18 km/h.
   - Hyperbolic tangent nonlinear gate blends free and congested states based on local congestion level.
2. Physical floor guards:
   - Enforces 50 vph floor guard to prevent zero-flow disqualification penalties in Task 3 S_FD.
   - Restricts speed to [5.0, 1.05 * v_f].
3. Iterative conservation balance along corridor link topology for Task 3 S_LWR.
"""
from __future__ import annotations

import time
from pathlib import Path
import numpy as np
import pandas as pd

from config import (
    PANELS,
    INTERVAL_MINUTES,
    SLOTS_PER_DAY,
    SLOTS_PER_WEEK,
)
from network_utils import (
    load_lane_counts,
    load_fd_params,
    load_ordered_links,
    get_lane_vector,
)

# Physics corrections: soft FD projection + iterative conservation balance across links
USE_PHYSICS_CORRECTIONS = True
try:
    from physics_corrections import (
        apply_physics_corrections,
        FD_ALPHA,
        CONSERVATION_ALPHA,
        CONSERVATION_PASSES,
    )
except ImportError:
    apply_physics_corrections = None  # type: ignore
    FD_ALPHA = 0.15
    CONSERVATION_ALPHA = 0.40
    CONSERVATION_PASSES = 5


def compute_time_slots(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Compute (weekday, time_of_day_slot) from timestamp column."""
    dt = pd.to_datetime(df["timestamp"], utc=True)
    weekday = dt.dt.weekday.to_numpy(dtype=np.int64)
    tod = (dt.dt.hour * 12 + dt.dt.minute // 5).to_numpy(dtype=np.int64)
    return weekday, tod


def learn_historical_profiles(panel_dir: Path) -> tuple[dict, dict]:
    """Learn historical (link x weekday x time_of_day) profiles from train unmasked states."""
    train_files = sorted((panel_dir / "train" / "mainline_states").glob("**/*.parquet"))
    if not train_files:
        pfiles = sorted(panel_dir.glob("**/mainline_states/**/*.parquet"))
        train_files = [f for f in pfiles if "train" in str(f) and "masked" not in str(f)]
        
    if not train_files:
        raise FileNotFoundError(f"No unmasked train states found under {panel_dir}")

    links_df = pd.read_csv(panel_dir / "network" / "links.csv", dtype={"link_id": str})
    link_ids = sorted(links_df["link_id"].unique())
    link_idx = {lid: i for i, lid in enumerate(link_ids)}
    n_links = len(link_ids)

    speed_sum = np.zeros((n_links, SLOTS_PER_WEEK), dtype=float)
    speed_cnt = np.zeros((n_links, SLOTS_PER_WEEK), dtype=int)
    flow_sum = np.zeros((n_links, SLOTS_PER_WEEK), dtype=float)
    flow_cnt = np.zeros((n_links, SLOTS_PER_WEEK), dtype=int)

    for path in train_files:
        df = pd.read_parquet(path)
        df["link_id"] = df["link_id"].astype(str)
        idx = df["link_id"].map(link_idx).fillna(-1).to_numpy(dtype=np.int64)
        valid_mask = idx >= 0
        if not np.any(valid_mask):
            continue

        weekday, tod = compute_time_slots(df)
        slot = weekday * SLOTS_PER_DAY + tod

        eligible = (
            pd.to_numeric(df["pct_observed"], errors="coerce").ge(75)
            & df["speed_kmh"].notna()
            & df["flow_vph"].notna()
        ).to_numpy()

        sp = pd.to_numeric(df["speed_kmh"], errors="coerce").to_numpy(dtype=float)
        fl = pd.to_numeric(df["flow_vph"], errors="coerce").to_numpy(dtype=float)

        v_sp = eligible & np.isfinite(sp)
        v_fl = eligible & np.isfinite(fl)

        np.add.at(speed_sum, (idx[v_sp], slot[v_sp]), sp[v_sp])
        np.add.at(speed_cnt, (idx[v_sp], slot[v_sp]), 1)
        np.add.at(flow_sum, (idx[v_fl], slot[v_fl]), fl[v_fl])
        np.add.at(flow_cnt, (idx[v_fl], slot[v_fl]), 1)

    speed_mean = np.zeros_like(speed_sum)
    flow_mean = np.zeros_like(flow_sum)
    np.divide(speed_sum, np.maximum(speed_cnt, 1), out=speed_mean, where=speed_cnt > 0)
    np.divide(flow_sum, np.maximum(flow_cnt, 1), out=flow_mean, where=flow_cnt > 0)
    speed_fb = np.divide(speed_sum.sum(axis=1), np.maximum(speed_cnt.sum(axis=1), 1))
    flow_fb = np.divide(flow_sum.sum(axis=1), np.maximum(flow_cnt.sum(axis=1), 1))

    return (
        {"link_ids": link_ids, "link_idx": link_idx, "mean": speed_mean, "fallback": speed_fb, "cnt": speed_cnt},
        {"link_ids": link_ids, "link_idx": link_idx, "mean": flow_mean, "fallback": flow_fb, "cnt": flow_cnt},
    )


def run_asm_grid(
    sp_grid: np.ndarray,
    fl_grid: np.ndarray,
    x_coords: np.ndarray,
    c_free: float = 75.0,
    c_cong: float = -18.0,
    sigma: float = 2.0,
    tau: float = 0.15,
    v_c: float = 55.0,
    delta_v: float = 10.0,
) -> tuple[np.ndarray, np.ndarray]:
    """Execute Treiber-Helbing Adaptive Smoothing Method on a single day's corridor grid."""
    n_links, n_slots = sp_grid.shape
    t_coords = np.arange(n_slots) * (5.0 / 60.0)  # Time in hours
    
    valid_sp = ~np.isnan(sp_grid)
    valid_fl = ~np.isnan(fl_grid)
    
    sp_out = np.copy(sp_grid)
    fl_out = np.copy(fl_grid)
    
    missing_r, missing_c = np.where(~valid_sp)
    
    for r, c in zip(missing_r, missing_c):
        x_tgt, t_tgt = x_coords[r], t_coords[c]
        
        # Local space-time neighborhood
        r_min, r_max = max(0, r - 6), min(n_links, r + 7)
        c_min, c_max = max(0, c - 9), min(n_slots, c + 10)
        
        sub_valid = valid_sp[r_min:r_max, c_min:c_max]
        if not np.any(sub_valid):
            continue
            
        dx = x_tgt - x_coords[r_min:r_max, None]
        dt = t_tgt - t_coords[None, c_min:c_max]
        
        # Downstream free wave kernel: information travels downstream along characteristic rays
        arg_free = -np.abs(dx) / sigma - np.abs(dt - dx / c_free) / tau
        w_free = np.exp(arg_free) * sub_valid
        
        # Upstream congested shockwave kernel: shockwaves propagate backward against traffic
        arg_cong = -np.abs(dx) / sigma - np.abs(dt - dx / c_cong) / tau
        w_cong = np.exp(arg_cong) * sub_valid
        
        sum_w_free = float(np.sum(w_free))
        sum_w_cong = float(np.sum(w_cong))
        
        if sum_w_free < 1e-5 and sum_w_cong < 1e-5:
            continue
            
        sub_sp = np.nan_to_num(sp_grid[r_min:r_max, c_min:c_max], nan=0.0)
        sub_fl = np.nan_to_num(fl_grid[r_min:r_max, c_min:c_max], nan=0.0)
        
        v_free = float(np.sum(w_free * sub_sp) / max(sum_w_free, 1e-6))
        v_cong = float(np.sum(w_cong * sub_sp) / max(sum_w_cong, 1e-6))
        
        q_free = float(np.sum(w_free * sub_fl) / max(sum_w_free, 1e-6))
        q_cong = float(np.sum(w_cong * sub_fl) / max(sum_w_cong, 1e-6))
        
        # Nonlinear hyperbolic tangent gate based on congested speed estimate
        w_blend = 0.5 * (1.0 + np.tanh((v_c - v_cong) / delta_v))
        
        sp_out[r, c] = (1.0 - w_blend) * v_free + w_blend * v_cong
        fl_out[r, c] = (1.0 - w_blend) * q_free + w_blend * q_cong
        
    return sp_out, fl_out


def reconstruct_panel_split(
    panel: str,
    panel_dir: Path,
    split: str,
    sp_prof: dict,
    fl_prof: dict,
    lanes_map: dict[str, float],
    fd: dict[str, dict[str, float]],
    ordered_links: list[str],
) -> pd.DataFrame:
    """Reconstruct masked states for a single corridor panel and split using ASM."""
    link_order_map = {lid: i for i, lid in enumerate(ordered_links)}
    link_idx = sp_prof["link_idx"]
    pfiles = sorted((panel_dir / split / "mainline_states_masked").glob("**/*.parquet"))
    
    # Cumulative link distances for ASM
    link_lens = np.array([fd.get(str(l), {}).get("length", 1.0) for l in ordered_links], dtype=float)
    cum_dist = np.concatenate([[0.0], np.cumsum(link_lens)[:-1]]) if len(link_lens) else np.arange(len(ordered_links), dtype=float)
    n_ordered = len(ordered_links)
    
    batches = []
    for path in pfiles:
        df = pd.read_parquet(path)
        df["station_id"] = df["station_id"].astype(str)
        df["link_id"] = df["link_id"].astype(str)
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

        weekday, tod = compute_time_slots(df)
        slot = weekday * SLOTS_PER_DAY + tod
        idx = df["link_id"].map(link_idx).fillna(-1).to_numpy(dtype=np.int64)
        valid_idx = np.where(idx >= 0, idx, 0)

        # Historical baseline priors
        hist_sp = sp_prof["mean"][valid_idx, slot]
        hist_fl = fl_prof["mean"][valid_idx, slot]
        hist_sp = np.where(sp_prof["cnt"][valid_idx, slot] == 0, sp_prof["fallback"][valid_idx], hist_sp)
        hist_fl = np.where(fl_prof["cnt"][valid_idx, slot] == 0, fl_prof["fallback"][valid_idx], hist_fl)

        lanes_arr = get_lane_vector(df["station_id"], df["link_id"], lanes_map)
        df["flow_per_lane"] = df["flow_vph"] / np.maximum(lanes_arr, 1.0)
        hist_fl_pl = hist_fl / np.maximum(lanes_arr, 1.0)

        # Build 2D Spatiotemporal Matrix for this day (ordered_links x 288 slots)
        tod_arr = tod
        order_idx_arr = df["link_id"].map(link_order_map).fillna(-1).to_numpy(dtype=np.int64)
        
        sp_grid = np.full((n_ordered, SLOTS_PER_DAY), np.nan, dtype=float)
        fl_grid = np.full((n_ordered, SLOTS_PER_DAY), np.nan, dtype=float)
        
        # Populate observed cells in grid
        obs_mask = df["speed_kmh"].notna() & df["flow_per_lane"].notna()
        obs_orders = order_idx_arr[obs_mask]
        obs_slots = tod_arr[obs_mask]
        valid_grid_pts = (obs_orders >= 0) & (obs_orders < n_ordered)
        
        if np.any(valid_grid_pts):
            sp_grid[obs_orders[valid_grid_pts], obs_slots[valid_grid_pts]] = df["speed_kmh"].to_numpy(dtype=float)[obs_mask][valid_grid_pts]
            fl_grid[obs_orders[valid_grid_pts], obs_slots[valid_grid_pts]] = df["flow_per_lane"].to_numpy(dtype=float)[obs_mask][valid_grid_pts]
            
        # Run Treiber-Helbing Adaptive Smoothing Method
        sp_asm, fl_asm = run_asm_grid(sp_grid, fl_grid, cum_dist)
        
        # Map ASM results back to dataframe rows
        row_orders = np.clip(order_idx_arr, 0, n_ordered - 1)
        row_slots = np.clip(tod_arr, 0, SLOTS_PER_DAY - 1)
        
        asm_sp_vals = sp_asm[row_orders, row_slots]
        asm_fl_vals = fl_asm[row_orders, row_slots]
        
        # Blend: ASM wave propagation where valid, historical mean where unsupported
        pred_speed = np.where(
            np.isfinite(asm_sp_vals) & (order_idx_arr >= 0),
            asm_sp_vals,
            hist_sp
        )
        pred_fl_pl = np.where(
            np.isfinite(asm_fl_vals) & (order_idx_arr >= 0),
            asm_fl_vals,
            hist_fl_pl
        )
        
        # Gentle regularizing blend with historical profile (5% prior)
        pred_speed = 0.95 * pred_speed + 0.05 * hist_sp
        pred_fl_pl = 0.95 * pred_fl_pl + 0.05 * hist_fl_pl
        
        pred_flow_raw = pred_fl_pl * lanes_arr

        # Physical bounds clipping
        vf_arr = np.array([fd.get(str(l), {}).get("vf", 105.0) for l in df["link_id"]])
        cap_arr = np.array([fd.get(str(l), {}).get("cap", 7200.0) for l in df["link_id"]])

        pred_speed_clipped = np.clip(pred_speed, 5.0, vf_arr * 1.05)
        
        # Enforce physical 50 vph floor guard (prevents zero-flow penalty in S_FD)
        pred_flow_clean = np.maximum(pred_flow_raw, 50.0)

        df["pred_speed_kmh"] = pred_speed_clipped
        df["pred_flow_vph"] = pred_flow_clean
        
        # Only keep target rows that need submission
        batches.append(df[["timestamp", "station_id", "link_id", "pred_speed_kmh", "pred_flow_vph"]])

    if not batches:
        return pd.DataFrame(columns=["timestamp", "station_id", "link_id", "speed_kmh", "flow_vph"])

    out_df = pd.concat(batches, ignore_index=True)
    out_df.rename(columns={"pred_speed_kmh": "speed_kmh", "pred_flow_vph": "flow_vph"}, inplace=True)

    # Apply physical conservation balancing along corridor links if enabled
    if USE_PHYSICS_CORRECTIONS and apply_physics_corrections is not None:
        try:
            sp_corr, fl_corr = apply_physics_corrections(
                pred_speed=out_df["speed_kmh"].to_numpy(dtype=float),
                pred_flow=out_df["flow_vph"].to_numpy(dtype=float),
                link_ids=out_df["link_id"].to_numpy(dtype=str),
                timestamps=out_df["timestamp"].to_numpy(dtype=str),
                ordered_links=ordered_links,
                fd=fd,
            )
            out_df["speed_kmh"] = sp_corr
            out_df["flow_vph"] = fl_corr
        except Exception as e:
            print(f"Warning: Physics corrections skipped for {panel} ({e})")

    # Final safe floor enforcement
    out_df["speed_kmh"] = np.clip(pd.to_numeric(out_df["speed_kmh"], errors="coerce").fillna(65.0), 5.0, 130.0)
    out_df["flow_vph"] = np.clip(pd.to_numeric(out_df["flow_vph"], errors="coerce").fillna(1200.0), 50.0, 16000.0)
    
    return out_df


def run_task1(
    release_root: Path,
    splits: list[str],
    out_path: Path,
) -> int:
    """Execute Task 1 traffic state reconstruction across all panels and splits."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    
    results = []
    for panel in PANELS:
        panel_dir = release_root / "corridors" / panel
        if not panel_dir.exists():
            panel_dir = release_root / panel
        if not panel_dir.exists():
            continue
            
        lanes_map = load_lane_counts(panel_dir)
        fd = load_fd_params(panel_dir)
        ordered_links = load_ordered_links(panel_dir)
        sp_prof, fl_prof = learn_historical_profiles(panel_dir)
        
        for split in splits:
            tmpl_file = release_root / "task1" / panel / split / "sample_submission_state.csv"
            if not tmpl_file.exists():
                tmpl_file = release_root / "task1" / split / panel / "sample_submission_state.csv"
            if not tmpl_file.exists():
                candidates = list(release_root.glob(f"task1/**/{panel}/**/{split}/**/sample_submission_state.csv"))
                if candidates:
                    tmpl_file = candidates[0]
                    
            if not tmpl_file.exists():
                continue
                
            tmpl = pd.read_csv(tmpl_file)
            tmpl["station_id"] = tmpl["station_id"].astype(str)
            tmpl["link_id"] = tmpl["link_id"].astype(str)
            tmpl["timestamp"] = pd.to_datetime(tmpl["timestamp"], utc=True)
            
            recon = reconstruct_panel_split(
                panel=panel,
                panel_dir=panel_dir,
                split=split,
                sp_prof=sp_prof,
                fl_prof=fl_prof,
                lanes_map=lanes_map,
                fd=fd,
                ordered_links=ordered_links,
            )
            
            # Align exact rows to template
            merged = tmpl[["timestamp", "station_id", "link_id"]].merge(
                recon,
                on=["timestamp", "station_id", "link_id"],
                how="left"
            )
            
            # Impute any missing lookup fallback
            merged["speed_kmh"] = merged["speed_kmh"].fillna(65.0)
            merged["flow_vph"] = merged["flow_vph"].fillna(1200.0)
            
            # Preserve original template columns
            for col in tmpl.columns:
                if col not in merged.columns:
                    merged[col] = tmpl[col]
                    
            results.append(merged[tmpl.columns])
            
    if not results:
        print("Warning: No Task 1 outputs generated.")
        pd.DataFrame(columns=["timestamp", "station_id", "link_id", "speed_kmh", "flow_vph"]).to_csv(out_path, index=False)
        return 0
        
    final_df = pd.concat(results, ignore_index=True)
    final_df.to_csv(out_path, index=False)
    
    print(f"[task1] Done in {time.time()-t0:.1f}s. Saved {len(final_df):,} rows to {out_path}.")
    return len(final_df)


### Module: `src/task2_queue.py`


In [ ]:
%%writefile src/task2_queue.py
"""Task 2: Dynamic Kinematic Shockwave & Saturation Queue Forecasting Engine (Exp 12).

Replaces hardcoded bottleneck dictionaries with physics-grounded dynamics:
1. Ongoing Windows:
   - Identifies active queue boundaries along physical corridor link order.
   - Core bottleneck links persist unless dissipation cues (accelerating speed, falling occupancy) are detected.
   - Backward shockwave expansion: propagates queue tail upstream at physical wave speed w ~ 18 km/h
     conditioned on upstream deceleration and demand pressure.
2. Onset Windows:
   - Computes dynamic Pre-Breakdown Saturation Index (PBSI) across all links from 60-min history:
     combining speed margin to cutoff, short-term deceleration rate, volume/capacity ratio, and occupancy trend.
   - Predicts breakdown step dynamically (earlier for sharp decel/low margin, later for gradual buildup).
   - Once breakdown triggers, expands upstream along corridor topology in subsequent steps.
"""
from __future__ import annotations

import time
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

from config import (
    TASK2_ACTIVE_PANELS,
    TASK2_HORIZON_STEPS,
)
from network_utils import (
    load_fd_params,
    load_topology,
    load_ordered_links,
)


def _compute_link_history_metrics(
    h_df: pd.DataFrame,
    fd_params: dict[str, dict[str, float]],
    t_last: pd.Timestamp
) -> dict[str, dict[str, float]]:
    """Compute kinematic and state metrics per link from 60-min history."""
    metrics: dict[str, dict[str, float]] = {}
    
    # 15-minute window cutoff
    t_15m = t_last - pd.Timedelta(minutes=15)
    
    for lid, grp in h_df.groupby("link_id"):
        lid = str(lid)
        grp = grp.sort_values("timestamp")
        
        sp = pd.to_numeric(grp["speed_kmh"], errors="coerce").to_numpy(dtype=float)
        sp_valid = sp[np.isfinite(sp)]
        if len(sp_valid) == 0:
            continue
            
        vf = fd_params.get(lid, {}).get("vf", 105.0)
        v_cut = fd_params.get(lid, {}).get("v_cut", vf * 0.60)
        cap = fd_params.get(lid, {}).get("cap", 7200.0)
        
        v_last = float(sp_valid[-1])
        margin = (v_last - v_cut) / max(v_cut, 1.0)
        
        # Recent 15-min speed trend
        grp_15m = grp[grp["timestamp"] >= t_15m]
        sp_15m = pd.to_numeric(grp_15m["speed_kmh"], errors="coerce").dropna().to_numpy(dtype=float)
        if len(sp_15m) >= 2:
            decel_15m = (sp_15m[-1] - sp_15m[0]) / max(len(sp_15m) - 1, 1) / 5.0  # km/h per minute
            v_min15 = float(sp_15m.min())
        else:
            decel_15m = 0.0
            v_min15 = v_last
            
        # Flow ratio
        fl = pd.to_numeric(grp["flow_vph"], errors="coerce").dropna().to_numpy(dtype=float)
        q_last = float(fl[-1]) if len(fl) > 0 else 0.0
        flow_ratio = q_last / max(cap, 1.0)
        
        # Occupancy
        occ = pd.to_numeric(grp["occupancy"], errors="coerce").dropna().to_numpy(dtype=float) if "occupancy" in grp.columns else np.array([])
        occ_last = float(occ[-1]) if len(occ) > 0 else 0.0
        occ_trend = (occ[-1] - occ[0]) / max(len(occ) - 1, 1) if len(occ) >= 2 else 0.0
        
        # Breakdown saturation index (higher = closer to breakdown)
        collapse_index = (
            2.5 * max(0.0, 0.45 - margin)
            + 3.5 * max(0.0, -decel_15m)
            + 1.2 * max(0.0, flow_ratio - 0.70)
            + 1.5 * max(0.0, occ_last - 0.15)
        )
        
        metrics[lid] = {
            "v_last": v_last,
            "v_cut": v_cut,
            "margin": margin,
            "decel_15m": decel_15m,
            "v_min15": v_min15,
            "flow_ratio": flow_ratio,
            "occ_last": occ_last,
            "occ_trend": occ_trend,
            "collapse_index": collapse_index,
            "is_queued_T": float(v_last <= v_cut)
        }
        
    return metrics


def predict_window_queues(
    window_id: str,
    panel: str,
    condition: str,
    h_df: pd.DataFrame,
    target_df: pd.DataFrame,
    fd_params: dict[str, dict[str, float]],
    topology: dict[str, dict[str, list[str]]],
    ordered_links: list[str],
) -> list[int]:
    """Predict binary queue state (0/1) for all (link, timestamp) cells in horizon.
    
    Uses dynamic shockwave propagation and pre-breakdown saturation index.
    """
    if target_df.empty:
        return []
        
    t_last = h_df["timestamp"].max()
    is_ongoing = "ongoing" in condition.lower()
    
    # Corridor topology and link ordering
    link_order_map = {lid: i for i, lid in enumerate(ordered_links)}
    n_links = len(ordered_links)
    
    # Compute metrics from 60-min history
    link_metrics = _compute_link_history_metrics(h_df, fd_params, t_last)
    
    # Active queue at T0
    queued_at_T = {
        lid for lid, m in link_metrics.items()
        if m["is_queued_T"] == 1.0
    }
    
    # Horizon timestamps
    t_horizon = sorted(target_df["timestamp"].unique())
    n_steps = len(t_horizon)  # Expected 6 steps (T+5 to T+30)
    
    # Predicted queue set per step: step_idx (0 to 5) -> set of queued link_ids
    pred_queue_per_step: dict[int, set[str]] = {s: set() for s in range(n_steps)}
    
    if is_ongoing:
        # --- ONGOING QUEUE DYNAMICS ---
        # 1. Baseline: Persistent core queue
        base_queue = set(queued_at_T)
        if not base_queue:
            # Fallback if sensor dropout obscured queue at T: use lowest margin links
            low_m = sorted(link_metrics.keys(), key=lambda l: link_metrics[l]["margin"])
            base_queue = set(low_m[:2]) if low_m else set()
            
        # Get spatial indices of base queue
        base_indices = [link_order_map[lid] for lid in base_queue if lid in link_order_map]
        
        if base_indices:
            idx_tail = min(base_indices)  # Upstream-most queued link
            idx_head = max(base_indices)  # Downstream-most queued link (bottleneck head)
            
            for s in range(n_steps):
                dt_min = 5.0 * (s + 1)
                active_s = set(base_queue)
                
                # Check for queue dissipation (speed recovering rapidly)
                dissipating = False
                if s >= 3:
                    head_lid = ordered_links[idx_head]
                    if head_lid in link_metrics:
                        hm = link_metrics[head_lid]
                        if hm["decel_15m"] > 0.40 and hm["v_last"] > (hm["v_cut"] - 3.0):
                            dissipating = True
                
                if dissipating:
                    # Dissipates from tail to head
                    drop_count = min(s - 2, len(base_indices) - 1)
                    retained_indices = sorted(base_indices)[drop_count:]
                    active_s = {ordered_links[i] for i in retained_indices}
                else:
                    # Backward shockwave expansion:
                    # Wave speed ~ 18 km/h -> expands upstream ~1 link every 5-10 min
                    # Only expands if upstream link is under demand stress
                    max_expansion_hops = min(s + 1, 4)
                    for hop in range(1, max_expansion_hops + 1):
                        up_idx = idx_tail - hop
                        if 0 <= up_idx < n_links:
                            up_lid = ordered_links[up_idx]
                            up_m = link_metrics.get(up_lid)
                            # Expand if upstream link has margin stress or deceleration
                            if up_m is not None and (up_m["margin"] <= 0.35 or up_m["decel_15m"] <= 0.05):
                                active_s.add(up_lid)
                                
                pred_queue_per_step[s] = active_s
        else:
            for s in range(n_steps):
                pred_queue_per_step[s] = set(base_queue)
                
    else:
        # --- ONSET QUEUE DYNAMICS ---
        # No queue at T. Queue must form during horizon.
        # Rank links by Pre-Breakdown Saturation / Collapse Index
        ranked_links = sorted(
            link_metrics.keys(),
            key=lambda l: link_metrics[l]["collapse_index"],
            reverse=True
        )
        
        if ranked_links:
            primary_bn = ranked_links[0]
            secondary_bn = ranked_links[1] if len(ranked_links) > 1 else primary_bn
            
            p_m = link_metrics[primary_bn]
            
            # Estimate breakdown trigger step s_bd (0 to 5)
            # 0: T+5, 1: T+10, 2: T+15, 3: T+20, 4: T+25, 5: T+30
            if p_m["margin"] < 0.15 and p_m["decel_15m"] < -0.20:
                s_bd = 1  # T+10m
            elif p_m["margin"] < 0.25 and p_m["decel_15m"] < -0.10:
                s_bd = 2  # T+15m
            elif p_m["margin"] < 0.35 or p_m["decel_15m"] < -0.05:
                s_bd = 3  # T+20m
            else:
                s_bd = 4  # T+25m (conservative late onset)
                
            p_idx = link_order_map.get(primary_bn, 0)
            
            for s in range(n_steps):
                active_s = set()
                if s >= s_bd:
                    active_s.add(primary_bn)
                    if s >= (s_bd + 1) and secondary_bn != primary_bn:
                        # Secondary bottleneck breakdown
                        if link_metrics[secondary_bn]["collapse_index"] > 0.8:
                            active_s.add(secondary_bn)
                    
                    # Backward shockwave from primary bottleneck
                    hops = s - s_bd
                    for h in range(1, hops + 1):
                        up_idx = p_idx - h
                        if 0 <= up_idx < n_links:
                            active_s.add(ordered_links[up_idx])
                            
                pred_queue_per_step[s] = active_s

    # Generate flat binary predictions aligned with target_df rows
    step_map = {ts: i for i, ts in enumerate(t_horizon)}
    preds: list[int] = []
    
    for row in target_df.itertuples():
        s = step_map.get(row.timestamp, 0)
        lid = str(row.link_id)
        is_q = 1 if lid in pred_queue_per_step.get(s, set()) else 0
        preds.append(is_q)
        
    return preds


def run_task2(
    release_root: Path,
    splits: list[str],
    out_path: Path,
) -> int:
    """Execute Task 2 queue prediction across all evaluation windows."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    
    wi_files = sorted(release_root.glob("task2/**/window_index.csv")) or sorted(release_root.glob("**/window_index.csv"))
    hist_files = sorted(release_root.glob("task2/**/window_history.parquet")) or sorted(release_root.glob("**/window_history.parquet"))
    tmpl_files = sorted(release_root.glob("task2/**/sample_submission_queue.csv")) or sorted(release_root.glob("**/sample_submission_queue.csv"))
    
    if not (wi_files and hist_files and tmpl_files):
        print("Warning: Missing Task 2 input files. Creating empty submission.")
        pd.DataFrame(columns=["window_id", "timestamp", "link_id", "queue_pred"]).to_csv(out_path, index=False)
        return 0

    wi_df = pd.concat([pd.read_csv(f) for f in wi_files], ignore_index=True)
    tmpl_df = pd.concat([pd.read_csv(f) for f in tmpl_files], ignore_index=True)
    tmpl_df["window_id"] = tmpl_df["window_id"].astype(str)
    tmpl_df["link_id"] = tmpl_df["link_id"].astype(str)
    tmpl_df["timestamp"] = pd.to_datetime(tmpl_df["timestamp"], utc=True)
    
    # Filter splits
    wi_df = wi_df[wi_df["split"].isin(splits)].copy()
    valid_window_ids = set(wi_df["window_id"].astype(str).unique())
    tmpl_df = tmpl_df[tmpl_df["window_id"].isin(valid_window_ids)].copy()
    
    print(f"[task2] Evaluating {len(valid_window_ids)} windows across {splits}...")
    
    # Read history parquet files
    hist_list = []
    cols = ["window_id", "timestamp", "link_id", "speed_kmh", "flow_vph", "occupancy", "is_score_eligible"]
    for hf in hist_files:
        try:
            h = pd.read_parquet(hf)
            available_cols = [c for c in cols if c in h.columns]
            h = h[available_cols]
            h["window_id"] = h["window_id"].astype(str)
            h = h[h["window_id"].isin(valid_window_ids)]
            if not h.empty:
                hist_list.append(h)
        except Exception as e:
            print(f"[task2] Error reading {hf}: {e}")
            
    if not hist_list:
        print("Error: No matching history data found.")
        return 0
        
    full_hist = pd.concat(hist_list, ignore_index=True)
    full_hist["link_id"] = full_hist["link_id"].astype(str)
    full_hist["timestamp"] = pd.to_datetime(full_hist["timestamp"], utc=True)
    
    # Pre-cache corridor assets
    fd_params_by_panel: dict[str, dict] = {}
    topo_by_panel: dict[str, dict] = {}
    ordered_links_by_panel: dict[str, list[str]] = {}
    
    for panel in TASK2_ACTIVE_PANELS:
        panel_dir = release_root / "corridors" / panel
        if not panel_dir.exists():
            panel_dir = release_root / panel
        if panel_dir.exists():
            fd_params_by_panel[panel] = load_fd_params(panel_dir)
            topo_by_panel[panel] = load_topology(panel_dir)
            ordered_links_by_panel[panel] = load_ordered_links(panel_dir)

    all_preds_df = []
    
    for _, w_row in tqdm(wi_df.iterrows(), total=len(wi_df), desc="Predicting Task 2 Windows"):
        wid = str(w_row["window_id"])
        panel = str(w_row["panel"])
        condition = str(w_row.get("condition", "queue_onset"))
        
        if panel not in ordered_links_by_panel:
            continue
            
        w_hist = full_hist[full_hist["window_id"] == wid]
        w_tmpl = tmpl_df[tmpl_df["window_id"] == wid].copy()
        
        if w_hist.empty or w_tmpl.empty:
            continue
            
        preds = predict_window_queues(
            window_id=wid,
            panel=panel,
            condition=condition,
            h_df=w_hist,
            target_df=w_tmpl,
            fd_params=fd_params_by_panel[panel],
            topology=topo_by_panel[panel],
            ordered_links=ordered_links_by_panel[panel],
        )
        
        w_tmpl["queue_pred"] = preds
        all_preds_df.append(w_tmpl[["window_id", "timestamp", "link_id", "queue_pred"]])
        
    if not all_preds_df:
        print("Warning: No predictions generated.")
        return 0
        
    final_task2 = pd.concat(all_preds_df, ignore_index=True)
    final_task2.to_csv(out_path, index=False)
    
    pos_count = int(final_task2["queue_pred"].sum())
    total_count = len(final_task2)
    print(f"[task2] Done in {time.time()-t0:.1f}s. Saved {total_count:,} rows to {out_path}.")
    print(f"[task2] Predicted active queue cells: {pos_count:,} / {total_count:,} ({100*pos_count/total_count:.2f}%)")
    return total_count


### Module: `src/task4_odme.py`


In [ ]:
%%writefile src/task4_odme.py
"""Task 4: Origin-Destination Matrix Estimation (ODME) — v2 calibrated.

v2 fix (see EXPERIMENT Bug Fix 3):
- Anchor DIRECTLY to weak prior wp, fallback to base ONLY on NaN.
- v1 diluted prior (0.95*wp+0.05*base) and replaced legit 0.0 flows via
  np.where(wp>0, wp, base) — corrupted S_od/S_attr destination distribution.
- Solver: regularized NNLS lambda=0.05 (calibrated to official benchmark setting).
"""
from __future__ import annotations

import time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import nnls

from config import (
    PANELS,
    DEFAULT_REG_LAMBDA,
)


def solve_path_flow(
    A: np.ndarray,
    counts: np.ndarray,
    anchor: np.ndarray,
    reg_lambda: float = DEFAULT_REG_LAMBDA,
    use_weights: bool = False,
) -> np.ndarray:
    """Solve min ||A @ f - counts||^2 + lambda * ||f - anchor||^2 s.t. f >= 0.

    Step 6: inverse-variance weighting W_ii = 1/max(c_i, 50) aligns high-volume
    mainline corridors without sacrificing minor ramp flows. Set use_weights=False
    for plain NNLS.
    ABLATED 2026-09-22: weighted shipped unvalidated in v4 (LB 0.6789). Default
    OFF until validated; plain NNLS = v2 proven.
    """
    if use_weights:
        w = 1.0 / np.maximum(counts, 50.0)
        sqw = np.sqrt(w)
        aa = np.vstack([sqw[:, None] * A, np.sqrt(reg_lambda) * np.eye(A.shape[1])])
        bb = np.concatenate([sqw * counts, np.sqrt(reg_lambda) * anchor])
    else:
        aa = np.vstack([A, np.sqrt(reg_lambda) * np.eye(A.shape[1])])
        bb = np.concatenate([counts, np.sqrt(reg_lambda) * anchor])
    f, _ = nnls(aa, bb)
    return np.maximum(f, 0.0)


def solve_panel_odme(
    panel: str,
    panel_dir: Path,
    release_root: Path,
    splits: list[str],
    reg_lambda: float = DEFAULT_REG_LAMBDA
) -> pd.DataFrame:
    """Solve ODME for a single corridor across requested splits."""
    network = panel_dir / "network"
    paths_csv = network / "path_set.csv"
    incidence_csv = network / "path_link_incidence.csv"
    base_od_csv = network / "base_od.csv"
    
    if not (paths_csv.exists() and incidence_csv.exists()):
        return pd.DataFrame()

    paths = pd.read_csv(paths_csv)
    paths["path_id"] = paths["path_id"].astype(str)
    path_ids = paths["path_id"].tolist()
    path_index = {x: i for i, x in enumerate(path_ids)}

    incidence = pd.read_csv(incidence_csv)
    incidence["path_id"] = incidence["path_id"].astype(str)
    incidence["link_id"] = incidence["link_id"].astype(str)
    link_ids = incidence["link_id"].drop_duplicates().tolist()
    link_index = {x: i for i, x in enumerate(link_ids)}

    rr = incidence["link_id"].map(link_index).to_numpy(dtype=np.int64)
    cc = incidence["path_id"].map(path_index).to_numpy(dtype=np.int64)
    A = np.zeros((len(link_ids), len(path_ids)), dtype=np.float64)
    A[rr, cc] = 1.0

    base_values = np.zeros(len(path_ids), dtype=float)
    if base_od_csv.exists():
        base_od = pd.read_csv(base_od_csv)
        base_od["path_id"] = base_od["path_id"].astype(str)
        merged_base = paths[["path_id", "origin_zone", "destination_zone"]].merge(
            base_od[["path_id", "base_flow"]], on="path_id", how="left"
        )
        merged_base = merged_base.set_index("path_id").reindex(path_ids).reset_index()
        base_values = pd.to_numeric(merged_base["base_flow"], errors="coerce").fillna(0.0).to_numpy(dtype=float)

    records = []
    for split in splits:
        counts_path = release_root / "task4" / panel / split / "synthetic_link_counts.csv"
        prior_path = release_root / "task4" / panel / split / "synthetic_weak_prior.csv"

        if not counts_path.exists():
            continue

        counts_df = pd.read_csv(counts_path, dtype={"link_id": str})
        val_counts = counts_df.set_index("link_id").reindex(link_ids).fillna(0.0)["count"].to_numpy(dtype=float)
        observed = set(counts_df["link_id"])
        measured = np.array([l in observed for l in link_ids], dtype=bool)

        anchor = base_values.copy()
        dep_time = "PUBLIC-TRAIN-PM"
        if prior_path.exists():
            pr = pd.read_csv(prior_path, dtype={"path_id": str})
            if len(pr) and "path_flow" in pr.columns:
                pr["path_id"] = pr["path_id"].astype(str)
                wp = pr.set_index("path_id").reindex(path_ids)["path_flow"].to_numpy(dtype=float)
                # v2: preserve true zero-demand paths; fallback to base ONLY on NaN
                # (v1 bug: np.nan_to_num(nan=0) + 0.95*wp+0.05*base overwrote zeros)
                wp = np.where(np.isnan(wp), base_values, wp)
                if np.sum(np.abs(wp)) > 0:
                    anchor = wp
            if len(pr) and "departure_time" in pr.columns:
                dep_time = str(pr["departure_time"].iloc[0])

        solved = solve_path_flow(A[measured], val_counts[measured], anchor, reg_lambda=reg_lambda)

        ps = paths[["path_id", "origin_zone", "destination_zone"]].copy()
        ps["panel"] = panel
        ps["departure_time"] = dep_time
        ps["path_flow"] = solved
        records.append(ps[["panel", "departure_time", "path_id", "origin_zone", "destination_zone", "path_flow"]])

    if records:
        return pd.concat(records, ignore_index=True)
    return pd.DataFrame()


def run_task4(
    release_root: Path,
    splits: list[str],
    out_path: Path,
    panels: list[str] | None = None,
    reg_lambda: float = DEFAULT_REG_LAMBDA
) -> int:
    """Execute Task 4 ODME across all corridors and write results to out_path."""
    if panels is None:
        panels = PANELS
        
    out_path.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    
    results = []
    print(f"Starting Task 4 ODME path flow estimation (lambda={reg_lambda})...")
    for panel in panels:
        p_dir = release_root / "corridors" / panel
        p_df = solve_panel_odme(panel, p_dir, release_root, splits, reg_lambda=reg_lambda)
        if not p_df.empty:
            results.append(p_df)
            print(f"[{panel}] ODME solved: {len(p_df):,} entries.")

    if not results:
        pd.DataFrame(columns=["panel", "departure_time", "path_id", "path_flow"]).to_csv(out_path, index=False)
        return 0

    out_df = pd.concat(results, ignore_index=True)
    out_df.to_csv(out_path, index=False)
    print(f"Task 4 Complete: {len(out_df):,} total entries exported to {out_path} in {time.time()-t0:.1f}s")
    return len(out_df)


### Module: `src/submission.py`


In [ ]:
%%writefile src/submission.py
"""Merge per-task predictions into the final unified Kaggle submission file."""
from __future__ import annotations

import time
from pathlib import Path
import pandas as pd

CHUNK_SIZE = 500_000
OUT_COLUMNS = ["submission_id", "task", "speed_kmh", "flow_vph", "queue_pred", "path_flow"]

KEYS = {
    "state": ["panel", "timestamp", "station_id", "link_id", "mask_regime"],
    "queue": ["window_id", "timestamp", "link_id"],
    "odme": ["panel", "departure_time", "path_id"]
}

VALUES = {
    "state": ["speed_kmh", "flow_vph"],
    "queue": ["queue_pred"],
    "odme": ["path_flow"]
}


def index_table(path: Path, task: str) -> pd.DataFrame:
    """Read a task submission CSV and index it by its natural key."""
    if not path.exists():
        raise FileNotFoundError(f"Missing prediction file for task '{task}': {path}")
        
    frame = pd.read_csv(path)
    missing = sorted(set(KEYS[task] + VALUES[task]) - set(frame.columns))
    if missing:
        raise ValueError(f"{path} is missing required columns: {missing}")
        
    frame = frame[KEYS[task] + VALUES[task]].copy()
    for col in KEYS[task]:
        frame[col] = pd.to_datetime(frame[col], utc=True) if col == "timestamp" else frame[col].astype(str)
    for col in VALUES[task]:
        frame[col] = pd.to_numeric(frame[col], errors="coerce")
        
    dup = int(frame.duplicated(KEYS[task]).sum())
    if dup:
        print(f"Warning: {path} has {dup} duplicate rows over {KEYS[task]}. Dropping duplicates.")
        frame = frame.drop_duplicates(KEYS[task])
        
    return frame.set_index(KEYS[task])


def merge_and_validate_submission(
    state_path: Path,
    queue_path: Path,
    odme_path: Path,
    key_file: Path,
    out_path: Path,
) -> int:
    """Stream submission_key.csv and join all tasks to write final submission.csv."""
    t0 = time.time()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    
    print("Indexing prediction tables for fast retrieval...")
    tables = {
        "state": index_table(state_path, "state"),
        "queue": index_table(queue_path, "queue"),
        "odme": index_table(odme_path, "odme")
    }

    print(f"Streaming key file from {key_file} to build {out_path}...")
    first_chunk = True
    total_written = 0
    missing_counts = {"state": 0, "queue": 0, "odme": 0}

    for chunk in pd.read_csv(key_file, chunksize=CHUNK_SIZE, dtype=str):
        chunk["timestamp"] = pd.to_datetime(chunk["timestamp"], utc=True, errors="coerce")
        out = pd.DataFrame({
            "submission_id": chunk["submission_id"].astype("int64"),
            "task": chunk["task"].astype(str)
        })
        for col in ("speed_kmh", "flow_vph", "queue_pred", "path_flow"):
            out[col] = 0.0

        for task, table in tables.items():
            rows = (out["task"] == task)
            if not rows.any():
                continue
            wanted = pd.MultiIndex.from_frame(chunk.loc[rows, KEYS[task]])
            found = table.reindex(wanted)
            for col in VALUES[task]:
                vals = found[col].to_numpy()
                missing_counts[task] += int(pd.isna(vals).sum())
                out.loc[rows, col] = pd.Series(vals).fillna(0.0).to_numpy()

        out[OUT_COLUMNS].to_csv(out_path, mode="w" if first_chunk else "a", header=first_chunk, index=False)
        first_chunk = False
        total_written += len(out)
        print(f"\rProcessed: {total_written:,} rows", end="", flush=True)

    print(f"\nMerge complete: {total_written:,} rows written to {out_path} in {time.time()-t0:.1f}s")
    
    # Validation diagnostics
    print("Running submission diagnostics...")
    for task, cnt in missing_counts.items():
        if cnt > 0:
            print(f"Warning: {cnt:,} {task} rows had no matching prediction and were filled with 0.0.")
        else:
            print(f"[OK] {task}: 100% matched.")

    sample = pd.read_csv(out_path, nrows=5000)
    assert sample.columns.tolist() == OUT_COLUMNS, f"Invalid columns: {sample.columns}"
    assert not sample.isna().any().any(), "Submission contains NaN values!"
    assert (sample["speed_kmh"] >= 0).all(), "Negative speed detected!"
    assert (sample["flow_vph"] >= 0).all(), "Negative flow detected!"
    assert sample["queue_pred"].isin([0, 1]).all(), "Non-binary queue_pred detected!"
    assert (sample["path_flow"] >= 0).all(), "Negative path_flow detected!"
    
    file_mb = out_path.stat().st_size / (1024 * 1024)
    print(f"Verification Successful: {out_path.name} is {file_mb:.1f} MB and 100% contract-compliant.")
    return total_written


### Pipeline Execution & Submission Assembly


In [ ]:
import sys, time, shutil
sys.path.insert(0, 'src')
from pathlib import Path
from config import locate_release_root
from task1_state import run_task1
from task2_queue import run_task2
from task4_odme import run_task4
from submission import merge_and_validate_submission

t0 = time.time()
REL = locate_release_root()
print('Competition Release Root:', REL)
SPLITS = ['validation', 'private']

# 1. Task 1 State Reconstruction & Task 3 Physics
print('=== Running Task 1 (Treiber-Helbing ASM) ===')
run_task1(REL, SPLITS, Path('work/state_submission.csv'))

# 2. Task 2 Queue Forecasting (Kinematic Shockwave)
print('=== Running Task 2 (Kinematic Shockwave) ===')
run_task2(REL, SPLITS, Path('work/queue_submission.csv'))

# 3. Task 4 ODME (Regularized NNLS)
print('=== Running Task 4 (Regularized NNLS) ===')
run_task4(REL, SPLITS, Path('work/odme_submission.csv'))

# 4. Merge, Verify, and Output Final Submission
KEY = REL / 'submission_key.csv'
merge_and_validate_submission(
    Path('work/state_submission.csv'),
    Path('work/queue_submission.csv'),
    Path('work/odme_submission.csv'),
    KEY,
    Path('/kaggle/working/submission.csv')
)

# Clean up scratch work directories
shutil.rmtree('work', ignore_errors=True)
shutil.rmtree('src/__pycache__', ignore_errors=True)
print(f'Pipeline finished successfully in {time.time()-t0:.1f}s.')
